# 🛠️ Aula 06 — Primeira Ferramenta
## Guilda de IA — Introdução à IA Generativa

<a href="https://colab.research.google.com/github/luksamuk/guilda-ia/blob/main/notebooks/aula06_primeira_ferramenta_colab.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Ferramentas: dando *poderes* ao agente.

---

## 🏗️ Setup

⚠️ Vá em `Runtime → Change runtime type` → **T4 GPU**

Execute a célula abaixo e avance — é boilerplate reutilizável.

In [ ]:
# ── Setup completo: Ollama + gemma4:e2b + warm up ────────────────
# Tudo em uma célula. Execute e prossiga.

# 1. Instalar Ollama + deps Python
!apt-get install -y zstd pciutils lshw > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q langchain langchain-openai langchain-core langgraph nest_asyncio requests

# 2. Workaround GPU Colab + keep alive
import os
os.environ["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"

# 3. Iniciar servidor
!pkill -f ollama 2> /dev/null; sleep 1
import subprocess
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env={**os.environ})

# 4. Aguardar servidor
import time, requests
for i in range(30):
    try:
        if requests.get("http://localhost:11434/api/tags", timeout=2).status_code == 200:
            break
    except:
        time.sleep(1)

# 5. Baixar modelo
!ollama pull gemma4:e2b

# 6. Warm up
print("🔥 Warm up...")
start = time.time()
!curl -s http://localhost:11434/api/chat -d '{"model":"gemma4:e2b","messages":[{"role":"user","content":"Oi"}],"stream":false,"keep_alive":-1}' > /dev/null
print(f"✅ Pronto em {time.time()-start:.1f}s")

## 1. Ferramentas com `@tool`

`@tool` transforma uma função Python em ferramenta pro LLM. **Pydantic** faz parsing e validação automático — sem `eval`, sem AST.

⚠️ **Importante:** nomes de funções e descrições em inglês. Modelos de linguagem entendem tool calling melhor em inglês — os valores dos argumentos e a resposta podem ser em português.

Usamos `get_word_length` em vez de `get_time` porque o Colab tem limitações de ambiente que impedem o acesso ao horário do sistema.

In [ ]:
from langchain_core.tools import tool

@tool
def calculate(a: float, b: float, operation: str) -> float:
    """Performs a math operation between two numbers and returns the exact result.
    Use when you need to do calculations.
    operation: 'add', 'subtract', 'multiply', 'divide'
    """
    ops = {'add': a+b, 'subtract': a-b, 'multiply': a*b, 'divide': a/b if b != 0 else 'Error: division by zero'}
    return ops.get(operation, f"Error: operation '{operation}' not supported")

@tool
def get_word_length(word: str) -> int:
    """Returns the number of characters in a word. Use when the user asks about the length of a word or how many letters it has."""
    return len(word)

print("✅ Ferramentas criadas:", [calculate.name, get_word_length.name])

## 2. Ciclo manual: `bind_tools`

`llm.bind_tools()` registra as ferramentas. O LLM **decide** qual chamar, mas **não executa** — isso fica por sua conta.

O Ollama fornece um endpoint compatível com a API da OpenAI. Com `ChatOpenAI` do `langchain-openai`, usamos esse endpoint **direto** — mesmo modelo local, sem enviar nada pra nuvem.

In [ ]:
from pprint import pprint
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gemma4:e2b",
    base_url="http://localhost:11434/v1",
    api_key="nao_precisa",  # Ollama local não usa chave
    temperature=0,
)

llm_com_ferramentas = llm.bind_tools([calculate, get_word_length])

# Passo 1: O LLM decide, mas NÃO executa
resposta = llm_com_ferramentas.invoke("Quanto é 234 vezes 987?")
print("Content:", repr(resposta.content))  # string vazia!
print("\nTool calls:")
pprint(resposta.tool_calls)

In [ ]:
# Passo 2: Executar a ferramenta
tc = resposta.tool_calls[0]
if tc['name'] == 'calculate':
    resultado = calculate.invoke(tc['args'])
elif tc['name'] == 'get_word_length':
    resultado = get_word_length.invoke(tc['args'])
print(f"Resultado da ferramenta: {resultado}")  # 230958.0

In [ ]:
# Passo 3: Resultado volta pro LLM
from langchain_core.messages import HumanMessage, ToolMessage

mensagens = [
    HumanMessage(content="Quanto é 234 vezes 987?"),
    resposta,  # AIMessage com tool_call
    ToolMessage(content=str(resultado), tool_call_id=tc['id']),
]

resposta_final = llm_com_ferramentas.invoke(mensagens)
print(resposta_final.content)

## 3. Ciclo automático: `create_agent`

O ciclo manual é bom pra aprender. Na prática, `create_agent` faz tudo automaticamente.

⚠️ **Mudança no LangChain 1.0+:** `create_agent` agora invoca com `{"messages": ...}` e retorna `resultado["messages"][-1].content`.

In [ ]:
from langchain.agents import create_agent

agente = create_agent(llm, [calculate, get_word_length])

# Uma linha — decidir, executar, responder
resultado = agente.invoke({"messages": "Quanto é 234 vezes 987?"})
print(resultado["messages"][-1].content)

In [ ]:
# Testar com comprimento de palavra e pergunta sem ferramenta
print(agente.invoke({"messages": "Quantas letras tem a palavra 'paralelepípedo'?"})["messages"][-1].content)
print()
print(agente.invoke({"messages": "Qual a capital da França?"})["messages"][-1].content)

## 4. Um gostinho de *async*

Todas as chamadas até agora usam `invoke()` — síncrono, o código espera.
Mas existe `ainvoke()` (com **a** de async) que não bloqueia:

Quem conhece JavaScript pode pensar em `ainvoke()` como uma **Promise** —
enquanto ela resolve, seu código pode fazer outras coisas.

⚠️ Por limitação de como notebooks Jupyter/Colab gerenciam o event loop,
precisamos de `nest_asyncio` para usar `asyncio.run()` dentro de uma célula.

In [ ]:
import asyncio
import nest_asyncio  # necessário em notebooks Jupyter/Colab
nest_asyncio.apply()  # permite asyncio.run() dentro do event loop existente

async def conversar():
    future = agente.ainvoke({"messages": "Quanto é 234 vezes 987?"})
    print("Processando...")  # roda imediatamente, sem esperar
    resultado = await future       # agora espera o resultado
    print(resultado["messages"][-1].content)

asyncio.run(conversar())

---
✅ **Resumo:**
- **`@tool`** transforma funções Python em ferramentas (Pydantic faz parsing)
- **`bind_tools`** → ciclo manual (entender o que acontece)
- **`create_agent`** → ciclo automático (produzir)
- O LLM *decide* qual ferramenta usar — você só registra
- ⚠️ Nomes e descrições de ferramentas em **inglês** — o modelo entende melhor